In [50]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3



load_dotenv(override=True)

openrouter_api = os.getenv("OPENROUTER_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")
if openrouter_api:
    print("Openrouter api found")
else:
    print("OPENROUTER NOT FOUND")
if groq_api:
    print("groq api found")
else:
    print("Groq api not found")

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key= groq_api)

groq_voice = OpenAI(base_url="https://api.groq.com/openai/v1/audio/speech", api_key = groq_api)

openrouter = OpenAI(base_url="https://openrouter.ai/api/v1" , api_key= openrouter_api)

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key= "ollama")

groq_model ="llama-3.3-70b-versatile"
openrouter_model = "openai/gpt-oss-120b:free"
ollama_model = "llama3.1:8b"

Openrouter api found
groq api found


In [2]:
system_message = """You are a helpful airline ticket booking assistant. 
You help customers find ticket prices to different cities.

RULES:
- You have access to a tool called get_ticket_price
- ALWAYS use the tool to get prices, never guess or make up prices
- NEVER show raw JSON or tool call syntax to the user
- Respond naturally and conversationally
- If asked about other cities, say they are not available
"""

In [3]:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of an airlines ticket to the desination city",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city":{
                "type":"string",
                "description":"The city that the customer wants to travel to",
            },
        },
        "required": ['destination_city'],
        'additionalProperties': False
    }
}

tools = [{'type':'function', "function": price_function}]

In [4]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [5]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [6]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

# FIXED FUNCTION

def chatt(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = groq.chat.completions.create(
        model=groq_model,
        messages=messages,
        tools=tools
    )

    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_calls(assistant_message)

        # ✅ Convert tool_calls Pydantic objects → plain dicts
        messages.append({
            "role": "assistant",
            "content": assistant_message.content or "",   # ✅ None → ""
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments
                    }
                }
                for tc in assistant_message.tool_calls    # ✅ serialized properly
            ]
        })

        messages.extend(tool_responses)                   # ✅ extend not append

        response = groq.chat.completions.create(
            model=groq_model,
            messages=messages,
            tools=tools
        )

    return response.choices[0].message.content

In [ ]:
def main():
    gr.ChatInterface(fn = chatt).launch()

if __name__ == "__main__":
    main()

* Running on local URL:  http://127.0.0.1:7885
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Sydney
DATABASE TOOL CALLED: Getting price for Tokyo


# IMAGE GENERATION MODEL

In [9]:
import base64
from io import BytesIO
from PIL import Image
from huggingface_hub import InferenceClient


In [18]:
from google import genai
from PIL import Image
from io import BytesIO

In [ ]:
gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
client = genai.Client(api_key=gemini_key)

In [29]:
# prompt = """
# Create a cinematic gourmet nano banana dessert
# served in a luxury futuristic restaurant
# with a glowing Gemini AI theme,
# blue and silver ambient lighting,
# ultra realistic food photography
# """

In [ ]:

# def artist(city):
#     response = client.models.generate_content(
#         model="models/gemini-3.1-flash-image-preview",
#         contents=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city},in a cartoon style"
#     )

#     # Process response
#     for part in response.candidates[0].content.parts:

#         # Text output
#         if hasattr(part, "text") and part.text:
#             print(part.text)

#         # Image output
#         if hasattr(part, "inline_data") and part.inline_data:
            
#             image_bytes = part.inline_data.data

#             image = Image.open(BytesIO(image_bytes))

#             image.save("generated_image.png")

#             print("Image saved as generated_image.png")

#             display("generated_image.png")

In [49]:
import sys
sys.path.insert(0, "/Users/nisumlimbu/mlx-examples/stable_diffusion")

from stable_diffusion import StableDiffusionXL
import mlx.core as mx
import numpy as np
from PIL import Image

sd = StableDiffusionXL("stabilityai/sdxl-turbo")

def artist(city, seed=42):
    prompt = f"A vacation in {city}, tourist spots and landmarks, cartoon style"
    output_path = f"{city}_vacation.png"

    print(f"Generating image for {city}...")

    latents = sd.generate_latents(
        prompt,
        n_images=1,
        cfg_weight=0.0,
        num_steps=4,
        seed=seed
    )

    for x_t in latents:
        mx.eval(x_t)

    decoded = sd.decode(x_t)
    mx.eval(decoded)

    image_array = (np.array(decoded[0]) * 255).astype(np.uint8)
    image = Image.fromarray(image_array)
    image.save(output_path)
    image.show()
    print(f"Saved: {output_path}")

In [57]:
artist("Paris")

Generating image for Paris...
Saved: Paris_vacation.png


In [64]:
def talker(message):
    response = groq.audio.speech.create(
      model="canopylabs/orpheus-v1-english",
      voice="autumn",   
      input=message,
      response_format="wav" 
    )
    return response.content

In [63]:
talker("HELLO")

BadRequestError: Error code: 400 - {'error': {'message': 'response_format must be one of [wav]', 'type': 'invalid_request_error'}}

In [65]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [66]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = groq.chat.completions.create(model=groq_model, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=groq_model, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [67]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Sydney
Generating image for Sydney...
Saved: Sydney_vacation.png


In [ ]:
# HF_TOKEN = ""  # ← your token

# client = InferenceClient(api_key=HF_TOKEN)

In [41]:
# def artist(city):
#     print("⏳ Generating image...")

#     image = client.text_to_image(
#         prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}",
#         model="wofmanaf/sd-knowledge-model-lora-sdxl-ft-text-encoder-freeze-unet-ep6",
#     )
#     display(image)

# # image.save("output.png")
# # print("✅ Saved as output.png"